In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# rutas y cache matplotlib para evitar errores de permisos
PROJECT_ROOT = Path().resolve().parent
MPL_DIR = PROJECT_ROOT / '.mplconfig'
MPL_DIR.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(MPL_DIR)

# permitir imports del paquete src/
sys.path.append(str(PROJECT_ROOT / 'src'))


In [ ]:

# Cargar datos de Barcelona y Madrid (Inside Airbnb)
bcn = pd.read_csv("../data/bcn_listings.csv")
mad = pd.read_csv("../data/madrid_listings.csv")


In [ ]:

def clean_price(series):
    """Convierte una columna de precio tipo '$120,00' a float"""
    series = series.str.replace("$", "", regex=False)
    series = series.str.replace(",", "", regex=False)
    return series.astype(float)


In [ ]:

# Limpieza básica
a_areas_bcn = ["Eixample", "Ciutat Vella", "Gràcia", "Sarrià-Sant Gervasi"]
a_areas_mad = ["Centro", "Salamanca", "Chamberí", "Retiro"]

bcn["price"] = clean_price(bcn["price"])
mad["price"] = clean_price(mad["price"])

# quedarse con pisos enteros de 2 a 6 personas en áreas céntricas
bcn_query = (
    "room_type == 'Entire home/apt' "
    "& 1 < accommodates < 7 "
    "& neighbourhood_group_cleansed in @a_areas_bcn"
)

mad_query = (
    "room_type == 'Entire home/apt' "
    "& 1 < accommodates < 7 "
    "& neighbourhood_group_cleansed in @a_areas_mad"
)

bcn["price_person"] = bcn["price"] / bcn["accommodates"]
mad["price_person"] = mad["price"] / mad["accommodates"]


In [ ]:

# Mediana del precio por persona según capacidad (2-6 pax)
bcn_summary = (
    bcn.query(bcn_query)
    .groupby("accommodates")["price_person"]
    .median()
    .reset_index(name="bcn_price_person")
)

mad_summary = (
    mad.query(mad_query)
    .groupby("accommodates")["price_person"]
    .median()
    .reset_index(name="mad_price_person")
)

summary = bcn_summary.merge(mad_summary, on="accommodates", how="inner")
summary


In [ ]:

# Gráfico comparando precio mediano por persona
plt.figure(figsize=(8, 5))
plt.plot(summary["accommodates"], summary["bcn_price_person"], marker="o", label="Barcelona")
plt.plot(summary["accommodates"], summary["mad_price_person"], marker="o", label="Madrid")
plt.title("Median price per person (2-6 guests)")
plt.xlabel("Guests")
plt.ylabel("Price per person (€)")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/graph3.png")
plt.show()

print("Gráfico guardado en ../reports/graph3.png")
